# 🎲 Synthetic Data Generator

**Feature Code**: SDG  
**Versão**: 2.0  
**Data**: 2026-04-04  
**Status**: ✅ 100% Independente

## Objetivo

Gerar dados sintéticos realistas de vendas regionais com **total independência de arquivos externos**. 

O metadata estatístico (distribuições, intervalos, relacionamentos) foi extraído dos dados reais originais (90 registros, jan-mai/2018) e está **pré-configurado no código**, garantindo:

- ✅ **Independência total**: Não requer nenhum arquivo externo
- ✅ **Distribuições estatísticas preservadas**: Regiões, seções, vendas
- ✅ **Relacionamentos consistentes**: Código-vendedor, mês-data
- ✅ **Volume configurável**: Gere quantos registros precisar
- ✅ **Reprodutibilidade**: Use random_seed para resultados determinísticos

## Como Usar

1. Configure os parâmetros na seção "Configuração de Parâmetros"
2. Execute todas as células
3. Os dados sintéticos serão gerados e exportados conforme configuração

## Parâmetros Disponíveis

- `n_registros`: Quantidade de registros a gerar (obrigatório)
- `random_seed`: Seed para reprodução (opcional, default=None)
- `output_format`: Formato de saída - "csv", "excel", ou "delta" (opcional, default="csv")

In [0]:
%run ./logger_control

In [0]:
%pip install openpyxl

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [0]:
# Implementa SPEC-SDG-R08: Logging Padronizado
logger = LogControl(
    logger_name="synthetic_data_generator",
    tbl_name="main.vendas_regionais.tb_logs_sdg"
)

logger.log_info("=" * 80)
logger.log_info("Inicializando Synthetic Data Generator (SDG)")
logger.log_info("Feature Code: SDG | Versão: 1.0")
logger.log_info("=" * 80)

## ⚙️ Configuração de Parâmetros

Configure abaixo os parâmetros para geração de dados sintéticos.

In [0]:
# Implementa SPEC-SDG-R02: Geração de Volume Configurável
# Implementa SPEC-SDG-R06: Reprodução Determinística

# PARÂMETRO OBRIGATÓRIO
n_registros = 1000  # Quantidade de registros a gerar

# PARÂMETROS OPCIONAIS
random_seed = None  # None = aleatório, int = reproduzível (ex: 42)
output_format = "csv"  # Opções: "csv", "excel", "delta"

logger.log_info(f"Parâmetros configurados: n_registros={n_registros}, random_seed={random_seed}, output_format={output_format}")

In [0]:
# Validar parâmetros
try:
    assert n_registros > 0, "n_registros deve ser maior que 0"
    assert output_format in ["csv", "excel", "delta"], "output_format deve ser 'csv', 'excel' ou 'delta'"
    logger.log_success("Parâmetros validados com sucesso")
except AssertionError as e:
    logger.log_error(f"Erro na validação de parâmetros: {str(e)}")
    logger.error_handler(e, debug_write_mode=False)
    raise

## 📊 Metadata Pré-configurado

Este gerador é **100% independente** de arquivos externos. O metadata estatístico foi extraído dos dados reais originais (90 registros de vendas jan-mai/2018) e está pré-configurado no código.

In [0]:
# Implementa SPEC-SDG-R01: Metadata Pré-configurado (100% independente de arquivo externo)
# Metadata extraído dos dados reais originais (90 registros, jan-mai/2018)

logger.log_info("="* 80)
logger.log_info("Carregando metadata pré-configurado")
logger.log_info("="* 80)

try:
    # Distribuições categóricas (extraídas dos dados originais)
    regiao_dist = {
        'Norte': 0.2333,      # ~21 registros
        'Sul': 0.2667,        # ~24 registros
        'Sudeste': 0.2556,    # ~23 registros
        'Nordeste': 0.2444    # ~22 registros
    }
    
    secao_dist = {
        'Eletrônicos': 0.1556,
        'Eletrodomésticos': 0.1111,
        'Móveis': 0.1333,
        'Informática': 0.1222,
        'Telefonia': 0.1333,
        'Games': 0.1222,
        'Livros': 0.1111,
        'Automotivo': 0.1111
    }
    
    # Vendedores (8 vendedores)
    vendedores = [
        'Roberto', 'Ricardo', 'Rodrigo', 'Roberta',
        'Renata', 'Rafael', 'Raquel', 'Ronaldo'
    ]
    
    # Regiões (4 regiões)
    regioes = ['Nordeste', 'Norte', 'Sudeste', 'Sul']
    
    # Seções (8 categorias)
    secoes = [
        'Automotivo', 'Eletrodomésticos', 'Eletrônicos', 'Games',
        'Informática', 'Livros', 'Móveis', 'Telefonia'
    ]
    
    # Mapeamento Código Vendedor → Vendedor (1:1)
    codigo_vendedor_map = {
        'Roberto': 1,
        'Ricardo': 2,
        'Rodrigo': 3,
        'Roberta': 4,
        'Renata': 5,
        'Rafael': 6,
        'Raquel': 7,
        'Ronaldo': 8
    }
    
    # Parâmetros da distribuição log-normal para vendas
    # (calculados dos dados originais: R$ 366,34 a R$ 19.228,10)
    vendas_log_mean = 8.655  # média do log(vendas)
    vendas_log_std = 0.782   # desvio padrão do log(vendas)
    vendas_min = 366.34
    vendas_max = 19228.10
    
    # Intervalo de datas (jan-mai/2018)
    data_min = pd.Timestamp('2018-01-03')
    data_max = pd.Timestamp('2018-05-31')
    
    # Armazenar metadata em dicionário
    metadata = {
        'regiao_dist': regiao_dist,
        'secao_dist': secao_dist,
        'vendedores': vendedores,
        'regioes': regioes,
        'secoes': secoes,
        'codigo_vendedor_map': codigo_vendedor_map,
        'vendas_log_mean': vendas_log_mean,
        'vendas_log_std': vendas_log_std,
        'vendas_min': vendas_min,
        'vendas_max': vendas_max,
        'data_min': data_min,
        'data_max': data_max
    }
    
    logger.log_success(f"Metadata pré-configurado carregado: {len(metadata)} atributos")
    logger.log_info(f"Regiões: {regioes}")
    logger.log_info(f"Vendedores: {len(vendedores)} vendedores")
    logger.log_info(f"Seções: {len(secoes)} categorias")
    logger.log_info(f"Intervalo de vendas: R$ {vendas_min:.2f} a R$ {vendas_max:.2f}")
    logger.log_info(f"Intervalo de datas: {data_min} a {data_max}")
    logger.log_info("="* 80)
    
except Exception as e:
    logger.log_error(f"Erro ao carregar metadata: {str(e)}")
    logger.error_handler(e, debug_write_mode=False)
    raise

## 🎲 Configuração de Aleatoriedade

In [0]:
# Implementa SPEC-SDG-R06: Reprodução Determinística
if random_seed is not None:
    np.random.seed(random_seed)
    logger.log_info(f"Random seed configurado: {random_seed} (reprodução determinística ativada)")
else:
    logger.log_info("Random seed não configurado (aleatoriedade total)")

## 🏭 Geração de Dados Sintéticos

Gerando {n_registros} registros sintéticos preservando distribuições originais.

In [0]:
# Implementa SPEC-SDG-R02: Geração de Volume Configurável
logger.log_info("=" * 80)
logger.log_info(f"Iniciando geração de {n_registros} registros sintéticos")
logger.log_info("=" * 80)

df_synth = pd.DataFrame()

In [0]:
# Implementa SPEC-SDG-R02: Geração de Volume Configurável
try:
    date_range_days = (metadata['data_max'] - metadata['data_min']).days
    random_days = np.random.randint(0, date_range_days + 1, size=n_registros)
    df_synth['Data da Venda'] = metadata['data_min'] + pd.to_timedelta(random_days, unit='D')
    
    logger.log_success(f"Coluna 'Data da Venda' gerada ({n_registros} registros)")
except Exception as e:
    logger.log_error("Erro ao gerar coluna 'Data da Venda'")
    logger.error_handler(e, debug_write_mode=False)
    raise

In [0]:
# Implementa SPEC-SDG-R05: Consistência de Relacionamentos
try:
    mes_map = {1: 'JAN', 2: 'FEV', 3: 'MAR', 4: 'ABR', 5: 'MAI', 6: 'JUN',
               7: 'JUL', 8: 'AGO', 9: 'SET', 10: 'OUT', 11: 'NOV', 12: 'DEZ'}
    df_synth['Mês'] = df_synth['Data da Venda'].dt.month.map(mes_map)
    
    logger.log_success("Coluna 'Mês' derivada automaticamente (consistência garantida)")
except Exception as e:
    logger.log_error("Erro ao derivar coluna 'Mês'")
    logger.error_handler(e, debug_write_mode=False)
    raise

In [0]:
# Implementa SPEC-SDG-R03: Preservação de Distribuições Categóricas
try:
    regioes = list(metadata['regiao_dist'].keys())
    probs = list(metadata['regiao_dist'].values())
    df_synth['Região'] = np.random.choice(regioes, size=n_registros, p=probs)
    
    logger.log_success("Coluna 'Região' gerada (distribuição original preservada)")
except Exception as e:
    logger.log_error("Erro ao gerar coluna 'Região'")
    logger.error_handler(e, debug_write_mode=False)
    raise

In [0]:
# Implementa SPEC-SDG-R02: Geração de Volume Configurável
try:
    df_synth['Vendedor'] = np.random.choice(metadata['vendedores'], size=n_registros)
    
    logger.log_success("Coluna 'Vendedor' gerada (distribuição uniforme)")
except Exception as e:
    logger.log_error("Erro ao gerar coluna 'Vendedor'")
    logger.error_handler(e, debug_write_mode=False)
    raise

In [0]:
# Implementa SPEC-SDG-R05: Consistência de Relacionamentos
try:
    df_synth['Código Vendedor'] = df_synth['Vendedor'].map(metadata['codigo_vendedor_map'])
    
    logger.log_success("Coluna 'Código Vendedor' mapeada (consistência garantida)")
except Exception as e:
    logger.log_error("Erro ao mapear coluna 'Código Vendedor'")
    logger.error_handler(e, debug_write_mode=False)
    raise

In [0]:
# Implementa SPEC-SDG-R03: Preservação de Distribuições Categóricas
try:
    secoes = list(metadata['secao_dist'].keys())
    probs_secao = list(metadata['secao_dist'].values())
    
    # Normalizar probabilidades para garantir soma = 1.0 (evita erros de arredondamento)
    probs_secao = np.array(probs_secao)
    probs_secao = probs_secao / probs_secao.sum()
    
    df_synth['Seção'] = np.random.choice(secoes, size=n_registros, p=probs_secao)
    
    logger.log_success("Coluna 'Seção' gerada (distribuição original preservada)")
except Exception as e:
    logger.log_error("Erro ao gerar coluna 'Seção'")
    logger.error_handler(e, debug_write_mode=False)
    raise

In [0]:
# Implementa SPEC-SDG-R04: Geração de Valores Numéricos Realistas
try:
    # Gerar valores com distribuição log-normal
    vendas_synth = np.random.lognormal(
        mean=metadata['vendas_log_mean'],
        sigma=metadata['vendas_log_std'],
        size=n_registros
    )
    
    # Clipar para intervalo observado
    vendas_synth = np.clip(vendas_synth, metadata['vendas_min'], metadata['vendas_max'])
    
    # Arredondar para 2 casas decimais
    df_synth['Vendas'] = np.round(vendas_synth, 2)
    
    logger.log_success(f"Coluna 'Vendas' gerada (distribuição log-normal, intervalo R$ {metadata['vendas_min']:.2f} a R$ {metadata['vendas_max']:.2f})")
except Exception as e:
    logger.log_error("Erro ao gerar coluna 'Vendas'")
    logger.error_handler(e, debug_write_mode=False)
    raise

## ✅ Validações de Qualidade

Validando integridade e consistência dos dados sintéticos gerados.

In [0]:
# Implementa validações de qualidade dos dados sintéticos
try:
    logger.log_info("=" * 80)
    logger.log_info("Executando validações de qualidade")
    logger.log_info("=" * 80)
    
    # Validação 1: Volume
    assert len(df_synth) == n_registros, f"Volume incorreto: esperado {n_registros}, obtido {len(df_synth)}"
    logger.log_success(f"✅ Validação 1: Volume correto ({n_registros} registros)")
    
    # Validação 2: Sem nulos
    nulls = df_synth.isnull().sum().sum()
    assert nulls == 0, f"Dados contêm {nulls} valores nulos"
    logger.log_success("✅ Validação 2: Sem valores nulos")
    
    # Validação 3: Consistência código-vendedor
    codigo_check = df_synth.groupby('Vendedor')['Código Vendedor'].nunique()
    assert (codigo_check == 1).all(), "Inconsistência código-vendedor detectada"
    logger.log_success("✅ Validação 3: Consistência código-vendedor OK")
    
    # Validação 4: Consistência mês-data
    mes_map = {1: 'JAN', 2: 'FEV', 3: 'MAR', 4: 'ABR', 5: 'MAI', 6: 'JUN',
               7: 'JUL', 8: 'AGO', 9: 'SET', 10: 'OUT', 11: 'NOV', 12: 'DEZ'}
    df_synth['_mes_calc'] = df_synth['Data da Venda'].dt.month.map(mes_map)
    assert (df_synth['Mês'] == df_synth['_mes_calc']).all(), "Inconsistência mês-data detectada"
    df_synth.drop(columns=['_mes_calc'], inplace=True)
    logger.log_success("✅ Validação 4: Consistência mês-data OK")
    
    # Validação 5: Intervalo de vendas
    assert df_synth['Vendas'].min() >= metadata['vendas_min'], "Vendas abaixo do mínimo observado"
    assert df_synth['Vendas'].max() <= metadata['vendas_max'], "Vendas acima do máximo observado"
    logger.log_success(f"✅ Validação 5: Intervalo de vendas OK (R$ {df_synth['Vendas'].min():.2f} a R$ {df_synth['Vendas'].max():.2f})")
    
    logger.log_info("=" * 80)
    logger.log_success("✅ Todas as validações passaram!")
    logger.log_info("=" * 80)
    
except AssertionError as e:
    logger.log_error(f"❌ Validação falhou: {str(e)}")
    logger.error_handler(e, debug_write_mode=False)
    raise
except Exception as e:
    logger.log_error("Erro durante validações")
    logger.error_handler(e, debug_write_mode=False)
    raise

## 💾 Export de Dados Sintéticos

Exportando dados gerados para o formato selecionado.

In [0]:
# Implementa SPEC-SDG-R07: Export Multi-Formato
try:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Path relativo ao notebook (em src/, subir 1 nível para vendas_regionais/, depois data/)
    data_dir = "../data"
    
    logger.log_info("=" * 80)
    logger.log_info(f"Exportando dados sintéticos (formato: {output_format})")
    logger.log_info(f"Destino: data/ (path relativo ao projeto)")
    logger.log_info("=" * 80)
    
    if output_format == 'excel':
        output_path = f"{data_dir}/dados_sinteticos_{timestamp}.xlsx"
        df_synth.to_excel(output_path, index=False, sheet_name="Dados Sintéticos")
        logger.log_success(f"✅ Dados exportados para Excel: data/dados_sinteticos_{timestamp}.xlsx")
        
    elif output_format == 'csv':
        output_path = f"{data_dir}/dados_sinteticos_{timestamp}.csv"
        df_synth.to_csv(output_path, index=False, encoding='utf-8-sig')
        logger.log_success(f"✅ Dados exportados para CSV: data/dados_sinteticos_{timestamp}.csv")
        
    elif output_format == 'delta':
        spark_df = spark.createDataFrame(df_synth)
        table_name = f"main.vendas_regionais.tb_dados_sinteticos_{timestamp}"
        spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        logger.log_success(f"✅ Dados exportados para Delta Table: {table_name}")
    
    else:
        logger.log_warning(f"⚠️ Formato desconhecido: {output_format}. Usando CSV como fallback.")
        output_path = f"{data_dir}/dados_sinteticos_{timestamp}.csv"
        df_synth.to_csv(output_path, index=False, encoding='utf-8-sig')
        logger.log_success(f"✅ Dados exportados para CSV: data/dados_sinteticos_{timestamp}.csv")
        
except Exception as e:
    logger.log_error(f"❌ Erro ao exportar dados sintéticos: {str(e)}")
    logger.error_handler(e, debug_write_mode=False)
    raise

## 🎉 Resumo Final

In [0]:
# Resumo final da execução
logger.log_info("=" * 80)
logger.log_info("🎉 SYNTHETIC DATA GENERATOR - EXECUÇÃO CONCLUÍDA COM SUCESSO!")
logger.log_info("=" * 80)
logger.log_info(f"📊 Registros gerados: {len(df_synth)}")
logger.log_info(f"📋 Colunas: {list(df_synth.columns)}")
logger.log_info(f"💾 Formato de export: {output_format}")
if random_seed is not None:
    logger.log_info(f"🎲 Random seed usado: {random_seed} (reproduzível)")
else:
    logger.log_info("🎲 Random seed: Não utilizado (aleatório)")
logger.log_info("=" * 80)
logger.log_success("✅ Todos os requisitos SDD implementados com sucesso!")
logger.log_info("=" * 80)

print("\n" + "="*80)
print("🎉 EXECUÇÃO CONCLUÍDA!")
print("="*80)
print(f"\n📊 Total de registros gerados: {len(df_synth)}")
print(f"💾 Formato de export: {output_format}")
print(f"\n🔍 Para visualizar os logs completos, consulte a tabela: main.vendas_regionais.tb_logs_sdg")
print("="*80)